In [2]:
# ============================================================================
# Moroccan benchmark — Mistral LLM inference runner
# Paste this entire file into ONE Jupyter cell and run it once.
#
# Required packages:
#     pip install pandas openpyxl requests tqdm
# ============================================================================

import os
import re
import time
import random
import threading
import getpass
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED

import pandas as pd
import requests
from tqdm import tqdm  # deliberately not tqdm.auto: avoids IProgress warnings


# ================================ CONFIG =====================================

DATA_DIR = Path.cwd() / "data eval equipe firdaws 16 juillet 2026"
INPUT_XLSX = DATA_DIR / "merged_eval_equipe_firdaws.xlsx"
OUTPUT_XLSX = DATA_DIR / "merged_eval_equipe_firdaws_WITH_RESPONSES.xlsx"

MODELS = [
    "mistral-large-2512",
    "mistral-medium-3-5",
    "mistral-small-2603",
    "ministral-14b-2512",
    "ministral-8b-2512",
]

API_URL = "https://api.mistral.ai/v1/conversations"

# Ten requests maximum at the same time.
MAX_WORKERS = 10

# Temperature 0 is recommended for reproducible evaluation.
# Change to 0.7 only if you explicitly want the sampling setting from your curl.
TEMPERATURE = 0.0
MAX_TOKENS = 2048
TOP_P = 1.0

# Network/retry settings.
CONNECT_TIMEOUT_SECONDS = 20
READ_TIMEOUT_SECONDS = 180
MAX_RETRIES = 7

# Resume from an existing output file and skip successful answers.
RESUME = True

# Save the entire workbook after this many newly completed API requests.
CHECKPOINT_EVERY = 50

# The API instructions field remains empty, exactly as in the provided curl.
# Task instructions are included directly in each user prompt.
API_INSTRUCTIONS = ""

SHEET_TASKS = {
    "qcm": "mcq",
    "binary questions": "binary",
    "open question": "open",
    "short answer": "short",
}

REQUIRED_COLUMNS = {
    "mcq": {"input", "response_A", "response_B", "response_C", "response_D"},
    "binary": {"input", "response_A", "response_B"},
    "open": {"input"},
    "short": {"input"},
}


# =============================== API KEY =====================================

API_KEY = "dnEaBGMpZrWGzXyOYokw6k2qDFKqkEtc"
if not API_KEY:
    API_KEY = getpass.getpass("Enter MISTRAL_API_KEY (hidden): ").strip()

if not API_KEY:
    raise RuntimeError(
        "No API key supplied. Set the MISTRAL_API_KEY environment variable "
        "or enter it when prompted."
    )


# ============================== VALIDATION ===================================

if not INPUT_XLSX.exists():
    raise FileNotFoundError(
        f"Input workbook not found:\n{INPUT_XLSX}\n\n"
        f"Current working directory is:\n{Path.cwd()}"
    )

DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input : {INPUT_XLSX}")
print(f"Output: {OUTPUT_XLSX}")
print(f"Parallel requests: {MAX_WORKERS}")


# ============================= SMALL HELPERS =================================

def clean(value) -> str:
    """Convert an Excel value into a clean string without turning NaN into 'nan'."""
    return "" if pd.isna(value) else str(value).strip()


def normalize_sheet_name(name: str) -> str:
    return re.sub(r"\s+", " ", str(name).strip().lower())


def task_of(sheet_name: str):
    key = normalize_sheet_name(sheet_name)

    if key in SHEET_TASKS:
        return SHEET_TASKS[key]

    # Tolerates minor differences in spaces/capitalization.
    compact = key.replace(" ", "")
    for expected_name, task in SHEET_TASKS.items():
        if expected_name.replace(" ", "") in compact:
            return task

    return None


def is_successful_response(value) -> bool:
    """True when a response exists and is not an error marker."""
    if not isinstance(value, str):
        return False
    value = value.strip()
    return bool(value) and not value.startswith("__ERROR__")


# ============================= PROMPT BUILDERS ===============================

MCQ_INSTRUCTION = (
    "اختر الجواب الصحيح من بين الاختيارات المعروضة. "
    "أجب فقط بحرف واحد: A أو B أو C أو D، دون شرح أو أي نص إضافي."
)

BINARY_INSTRUCTION = (
    "اختر الجواب الصحيح من بين الخيارين المعروضين. "
    "أجب فقط بحرف واحد: A أو B، دون شرح أو أي نص إضافي."
)

OPEN_INSTRUCTION = (
    "أجب عن السؤال بوضوح ودقة وبنفس لغة السؤال. "
    "قدّم جواباً مركزاً ومفيداً، دون الخروج عن الموضوع."
)

SHORT_INSTRUCTION = (
    "أجب بجواب قصير ودقيق جداً وبنفس لغة السؤال. "
    "استخدم كلمة واحدة أو تعبيراً واحداً أو اسماً أو مكاناً أو تاريخاً فقط، "
    "دون شرح إضافي متى كان ذلك ممكناً."
)


def build_prompt(task: str, row: pd.Series) -> str:
    """Build the complete user prompt. Ground-truth columns are never included."""
    question = clean(row.get("input"))

    if task == "mcq":
        options = []
        for letter in ("A", "B", "C", "D"):
            option = clean(row.get(f"response_{letter}"))
            if option:
                options.append(f"{letter}. {option}")

        return (
            f"{MCQ_INSTRUCTION}\n\n"
            f"السؤال / Question:\n{question}\n\n"
            f"الاختيارات / Choices:\n" + "\n".join(options) +
            "\n\nالجواب / Answer:"
        )

    if task == "binary":
        options = []
        for letter in ("A", "B"):
            option = clean(row.get(f"response_{letter}"))
            if option:
                options.append(f"{letter}. {option}")

        return (
            f"{BINARY_INSTRUCTION}\n\n"
            f"السؤال / Question:\n{question}\n\n"
            f"الاختيارات / Choices:\n" + "\n".join(options) +
            "\n\nالجواب / Answer:"
        )

    if task == "short":
        return f"{SHORT_INSTRUCTION}\n\nالسؤال / Question:\n{question}\n\nالجواب / Answer:"

    if task == "open":
        return f"{OPEN_INSTRUCTION}\n\nالسؤال / Question:\n{question}\n\nالجواب / Answer:"

    raise ValueError(f"Unsupported task: {task}")


# ================================ API CALL ===================================

_thread_local = threading.local()


def get_session() -> requests.Session:
    """Create one requests.Session per worker thread."""
    if not hasattr(_thread_local, "session"):
        session = requests.Session()
        session.headers.update(
            {
                "Content-Type": "application/json",
                "Authorization": f"Bearer {API_KEY}",
            }
        )
        _thread_local.session = session

    return _thread_local.session


def extract_text(payload) -> str:
    """
    Extract assistant text from the Conversations API response.
    Handles string content, text chunks, and chat-completion-like fallbacks.
    """
    texts = []

    def visit(content):
        if isinstance(content, str):
            if content.strip():
                texts.append(content.strip())
            return

        if isinstance(content, list):
            for item in content:
                visit(item)
            return

        if not isinstance(content, dict):
            return

        item_type = content.get("type")

        # Keep final textual chunks, but do not collect thinking/reasoning chunks.
        if isinstance(content.get("text"), str) and item_type in (
            None,
            "text",
            "output_text",
            "message.output",
        ):
            text = content["text"].strip()
            if text:
                texts.append(text)
            return

        # Some payload shapes nest the actual text under "content".
        if "content" in content:
            visit(content["content"])

    if isinstance(payload, dict):
        # Native Conversations API shape.
        outputs = payload.get("outputs") or []
        for output in outputs:
            if not isinstance(output, dict):
                continue

            output_type = output.get("type")
            role = output.get("role")

            if output_type in (None, "message.output") or role in (None, "assistant"):
                visit(output.get("content"))

        # Fallback for Chat Completions-like responses.
        if not texts:
            for choice in payload.get("choices") or []:
                if isinstance(choice, dict):
                    message = choice.get("message") or {}
                    if isinstance(message, dict):
                        visit(message.get("content"))

        # Last-resort payload shapes.
        if not texts:
            visit(payload.get("content"))
        if not texts:
            visit(payload.get("output_text"))

    return "\n".join(texts).strip()


def response_excerpt(response: requests.Response, limit: int = 500) -> str:
    try:
        body = response.json()
        text = str(body)
    except Exception:
        text = response.text
    return text.replace("\n", " ")[:limit]


def retry_delay(response, attempt: int) -> float:
    """Respect Retry-After when present; otherwise use exponential backoff + jitter."""
    if response is not None:
        retry_after = response.headers.get("Retry-After")
        if retry_after:
            try:
                return min(float(retry_after), 60.0)
            except ValueError:
                pass

    return min(2 ** attempt, 30) + random.uniform(0.0, 1.0)


def call_model(model: str, prompt: str) -> str:
    payload = {
        "model": model,
        "inputs": [{"role": "user", "content": prompt}],
        "tools": [],
        "completion_args": {
            "temperature": TEMPERATURE,
            "max_tokens": MAX_TOKENS,
            "top_p": TOP_P,
        },
        "instructions": API_INSTRUCTIONS,
        # This benchmark does not need thousands of persistent conversation records.
        "store": False,
        "stream": False,
    }

    last_error = "unknown error"
    transient_statuses = {408, 409, 425, 429, 500, 502, 503, 504}

    for attempt in range(MAX_RETRIES):
        response = None

        try:
            response = get_session().post(
                API_URL,
                json=payload,
                timeout=(CONNECT_TIMEOUT_SECONDS, READ_TIMEOUT_SECONDS),
            )

            if response.status_code in transient_statuses:
                last_error = (
                    f"HTTP {response.status_code}: {response_excerpt(response)}"
                )
            elif not response.ok:
                # Non-transient errors such as 400/401/403/404/422 should not be retried.
                return (
                    f"__ERROR__: HTTP {response.status_code}: "
                    f"{response_excerpt(response)}"
                )
            else:
                try:
                    data = response.json()
                except ValueError:
                    last_error = f"Invalid JSON response: {response.text[:500]}"
                else:
                    text = extract_text(data)
                    if text:
                        return text
                    last_error = f"Empty model response. Payload: {str(data)[:500]}"

        except requests.Timeout as exc:
            last_error = f"timeout: {exc}"
        except requests.ConnectionError as exc:
            last_error = f"connection error: {exc}"
        except requests.RequestException as exc:
            last_error = f"request error: {exc}"
        except Exception as exc:
            last_error = f"unexpected error: {type(exc).__name__}: {exc}"

        if attempt < MAX_RETRIES - 1:
            time.sleep(retry_delay(response, attempt))

    return f"__ERROR__: {last_error}"


# ================================ PARSING ====================================

ARABIC_TO_LATIN_CHOICE = {
    "أ": "A",
    "ا": "A",
    "ب": "B",
    "ج": "C",
    "د": "D",
}


def normalize_option_text(text: str) -> str:
    text = clean(text).casefold()
    text = re.sub(r"[`*_#]", "", text)
    text = re.sub(r"[\s\.,،;؛:!?؟\-–—()\[\]{}\"'«»]+", " ", text)
    return text.strip()


def parse_choice(text, row=None, valid=("A", "B", "C", "D")) -> str:
    """
    Parse A/B/C/D from a raw response.
    If the model returns the complete option text instead of a letter,
    map that exact option text back to its letter.
    """
    if not isinstance(text, str) or text.startswith("__ERROR__"):
        return ""

    original = text.strip()
    if not original:
        return ""

    # Remove simple Markdown/code fencing and surrounding quotes.
    candidate = re.sub(r"^```(?:text)?\s*|\s*```$", "", original, flags=re.I).strip()
    candidate = candidate.strip(" \t\r\n\"'`*_")

    # Convert isolated Arabic option letters to Latin letters.
    converted = candidate
    for arabic_letter, latin_letter in ARABIC_TO_LATIN_CHOICE.items():
        converted = re.sub(
            rf"(?<![\w\u0600-\u06FF]){arabic_letter}(?![\w\u0600-\u06FF])",
            latin_letter,
            converted,
        )

    upper = converted.upper()
    allowed = "".join(valid)

    patterns = [
        rf"^\s*[\(\[]?([{allowed}])[\)\].:،,\-\s]*$",
        rf"^\s*[\(\[]?([{allowed}])[\)\].:،,\-\s]",
        rf"(?:ANSWER|RÉPONSE|REPONSE|CHOICE|OPTION|الجواب|الإجابة|الخيار)"
        rf"\s*[:：\-]?\s*[\(\[]?([{allowed}])(?:\b|\s|[\)\].،,])",
        rf"\b([{allowed}])\b",
    ]

    for pattern in patterns:
        match = re.search(pattern, upper, flags=re.MULTILINE)
        if match:
            return match.group(1)

    # Fallback: exact match with the text of one displayed option.
    if row is not None:
        normalized_answer = normalize_option_text(original)
        for letter in valid:
            option = normalize_option_text(row.get(f"response_{letter}"))
            if option and normalized_answer == option:
                return letter

    return ""


def normalize_gold_choice(value, valid) -> str:
    """Normalize a ground-truth cell that normally contains A/B/C/D."""
    parsed = parse_choice(clean(value), valid=valid)
    return parsed or clean(value).upper()[:1]


# ============================== LOAD WORKBOOK ================================

source_excel = pd.ExcelFile(INPUT_XLSX)
print("\nSheets found:", source_excel.sheet_names)

sheets = {}

for sheet_name in source_excel.sheet_names:
    task = task_of(sheet_name)
    df = pd.read_excel(source_excel, sheet_name=sheet_name)
    df.columns = [str(column).strip() for column in df.columns]

    if task is None:
        print(f"  ! {sheet_name}: unrecognized; it will be copied unchanged")
        sheets[sheet_name] = (df, None)
        continue

    missing = REQUIRED_COLUMNS[task] - set(df.columns)
    if missing:
        raise ValueError(
            f"Sheet '{sheet_name}' is recognized as task '{task}' but is missing "
            f"these columns: {sorted(missing)}"
        )

    print(f"  - {sheet_name}: task={task}, rows={len(df)}")
    sheets[sheet_name] = (df, task)


# Resume previously saved model columns when possible.
if RESUME and OUTPUT_XLSX.exists():
    try:
        previous_excel = pd.ExcelFile(OUTPUT_XLSX)

        for sheet_name, (df, task) in sheets.items():
            if sheet_name not in previous_excel.sheet_names:
                continue

            previous_df = pd.read_excel(
                previous_excel,
                sheet_name=sheet_name,
                dtype=object,
                keep_default_na=False,
            )
            previous_df.columns = [str(column).strip() for column in previous_df.columns]

            if len(previous_df) != len(df):
                print(
                    f"  ! Cannot resume sheet '{sheet_name}': row count changed "
                    f"({len(previous_df)} previously, {len(df)} now)."
                )
                continue

            for model in MODELS:
                for suffix in ("_response", "_parsed"):
                    column = f"{model}{suffix}"
                    if column in previous_df.columns:
                        df[column] = previous_df[column].values

        print(f"\nResumed available answers from:\n{OUTPUT_XLSX}")

    except Exception as exc:
        print(
            f"\nWARNING: Existing output could not be read, so this run will start "
            f"from the source workbook.\nReason: {exc}"
        )


# ============================= SAFE EXCEL SAVE ================================

def save_all() -> None:
    """
    Atomically save all sheets.

    Important: the temporary file still ends in .xlsx, which openpyxl requires.
    """
    temp_path = OUTPUT_XLSX.with_name(
        f"{OUTPUT_XLSX.stem}.__temporary__.xlsx"
    )

    try:
        if temp_path.exists():
            temp_path.unlink()

        with pd.ExcelWriter(temp_path, engine="openpyxl") as writer:
            for sheet_name, (df, _) in sheets.items():
                df.to_excel(writer, sheet_name=sheet_name, index=False)

        # Atomic replacement on the same filesystem.
        os.replace(temp_path, OUTPUT_XLSX)

    except PermissionError as exc:
        try:
            if temp_path.exists():
                temp_path.unlink()
        except Exception:
            pass

        raise PermissionError(
            f"Could not write '{OUTPUT_XLSX}'. Close the workbook in Excel or any "
            f"other program, then rerun the cell."
        ) from exc

    except Exception:
        try:
            if temp_path.exists():
                temp_path.unlink()
        except Exception:
            pass
        raise


# =========================== BOUNDED PARALLEL RUN =============================

def ensure_model_columns(df: pd.DataFrame, task: str, model: str):
    response_column = f"{model}_response"
    if response_column not in df.columns:
        df[response_column] = pd.NA

    parsed_column = None
    if task in ("mcq", "binary"):
        parsed_column = f"{model}_parsed"
        if parsed_column not in df.columns:
            df[parsed_column] = ""

        # Recompute missing parsed values for any responses loaded during resume.
        valid = ("A", "B", "C", "D") if task == "mcq" else ("A", "B")
        for index in df.index:
            raw = df.at[index, response_column]
            current_parsed = clean(df.at[index, parsed_column])
            if is_successful_response(raw) and not current_parsed:
                df.at[index, parsed_column] = parse_choice(
                    raw,
                    row=df.loc[index],
                    valid=valid,
                )

    return response_column, parsed_column


def run_model_on_sheet(model: str, sheet_name: str, df: pd.DataFrame, task: str):
    response_column, parsed_column = ensure_model_columns(df, task, model)

    valid = None
    if task == "mcq":
        valid = ("A", "B", "C", "D")
    elif task == "binary":
        valid = ("A", "B")

    # Never send empty rows to the API.
    empty_input_indices = [
        index for index in df.index if not clean(df.at[index, "input"])
    ]
    for index in empty_input_indices:
        df.at[index, response_column] = ""
        if parsed_column:
            df.at[index, parsed_column] = ""

    todo = [
        index
        for index in df.index
        if clean(df.at[index, "input"])
        and not (RESUME and is_successful_response(df.at[index, response_column]))
    ]

    if not todo:
        print(f"[{model}] {sheet_name}: already complete; skipping")
        return

    print(f"[{model}] {sheet_name}: {len(todo)} rows to run")

    # Only MAX_WORKERS futures are submitted at any moment. This prevents a
    # checkpoint error or interruption from leaving hundreds of queued calls.
    executor = ThreadPoolExecutor(max_workers=MAX_WORKERS)
    pending = iter(todo)
    in_flight = {}
    newly_completed = 0
    aborted = False

    def fill_available_slots():
        while len(in_flight) < MAX_WORKERS:
            try:
                index = next(pending)
            except StopIteration:
                break

            prompt = build_prompt(task, df.loc[index])
            future = executor.submit(call_model, model, prompt)
            in_flight[future] = index

    try:
        fill_available_slots()

        with tqdm(
            total=len(todo),
            desc=f"{model} | {sheet_name}",
            unit="row",
            dynamic_ncols=True,
        ) as progress:
            while in_flight:
                finished, _ = wait(
                    tuple(in_flight.keys()),
                    return_when=FIRST_COMPLETED,
                )

                for future in finished:
                    index = in_flight.pop(future)

                    try:
                        raw_response = future.result()
                    except Exception as exc:
                        raw_response = (
                            f"__ERROR__: {type(exc).__name__}: {exc}"
                        )

                    df.at[index, response_column] = raw_response

                    if parsed_column:
                        df.at[index, parsed_column] = parse_choice(
                            raw_response,
                            row=df.loc[index],
                            valid=valid,
                        )

                    newly_completed += 1
                    progress.update(1)

                    if newly_completed % CHECKPOINT_EVERY == 0:
                        save_all()
                        progress.set_postfix_str("checkpoint saved")

                fill_available_slots()

    except BaseException:
        aborted = True

        for future in in_flight:
            future.cancel()

        # Best-effort checkpoint without hiding the original exception.
        try:
            save_all()
            print("\nCheckpoint saved before stopping.")
        except Exception as save_error:
            print(f"\nWARNING: emergency checkpoint failed: {save_error}")

        raise

    finally:
        if aborted:
            # Do not wait for a long queue; at most ten active requests exist.
            executor.shutdown(wait=False, cancel_futures=True)
        else:
            executor.shutdown(wait=True)

    save_all()


# Save once before API execution so the output workbook and response columns
# begin to exist as soon as each model is initialized.
for model in MODELS:
    for _, (df, task) in sheets.items():
        if task is not None:
            ensure_model_columns(df, task, model)
save_all()


# ================================= RUN ========================================

for model in MODELS:
    print("\n" + "=" * 90)
    print(f"MODEL: {model}")
    print("=" * 90)

    for sheet_name, (df, task) in sheets.items():
        if task is None:
            continue

        run_model_on_sheet(model, sheet_name, df, task)

save_all()
print(f"\nDONE. Workbook written to:\n{OUTPUT_XLSX}")


# ============================= SANITY REPORT =================================

report_rows = []

for sheet_name, (df, task) in sheets.items():
    if task is None:
        continue

    for model in MODELS:
        response_column = f"{model}_response"
        parsed_column = f"{model}_parsed"

        if response_column not in df.columns:
            continue

        response_values = df[response_column]
        response_strings = response_values.fillna("").astype(str)

        record = {
            "sheet": sheet_name,
            "task": task,
            "model": model,
            "rows": len(df),
            "successful": int(response_strings.apply(is_successful_response).sum()),
            "errors": int(response_strings.str.startswith("__ERROR__").sum()),
            "empty": int((response_strings.str.strip() == "").sum()),
        }

        if task in ("mcq", "binary") and parsed_column in df.columns:
            valid = ("A", "B", "C", "D") if task == "mcq" else ("A", "B")
            parsed = df[parsed_column].fillna("").astype(str).str.strip().str.upper()

            record["unparsed"] = int(
                (
                    response_strings.apply(is_successful_response)
                    & (parsed == "")
                ).sum()
            )

            if "correct_answer" in df.columns:
                gold = df["correct_answer"].apply(
                    lambda value: normalize_gold_choice(value, valid)
                )
                usable = gold.isin(valid)
                record["accuracy"] = (
                    round(float((parsed[usable] == gold[usable]).mean()), 4)
                    if usable.any()
                    else None
                )

        report_rows.append(record)

report = pd.DataFrame(report_rows)

try:
    display(report)
except NameError:
    print(report.to_string(index=False))


Input : C:\Users\Abderrahman\Desktop\moderation llm\data eval equipe firdaws 16 juillet 2026\merged_eval_equipe_firdaws.xlsx
Output: C:\Users\Abderrahman\Desktop\moderation llm\data eval equipe firdaws 16 juillet 2026\merged_eval_equipe_firdaws_WITH_RESPONSES.xlsx
Parallel requests: 10

Sheets found: ['QCM', 'Binary questions', 'Open Question', 'Short answer']
  - QCM: task=mcq, rows=589
  - Binary questions: task=binary, rows=245
  - Open Question: task=open, rows=606
  - Short answer: task=short, rows=296

MODEL: mistral-large-2512
[mistral-large-2512] QCM: 588 rows to run


mistral-large-2512 | QCM: 100%|███████████████████████████████████| 588/588 [00:53<00:00, 10.95row/s, checkpoint saved]


[mistral-large-2512] Binary questions: 236 rows to run


mistral-large-2512 | Binary questions: 100%|██████████████████████| 236/236 [00:23<00:00,  9.84row/s, checkpoint saved]


[mistral-large-2512] Open Question: 606 rows to run


mistral-large-2512 | Open Question: 100%|█████████████████████████| 606/606 [14:19<00:00,  1.42s/row, checkpoint saved]


... (32 outputs left out)

[ministral-8b-2512] Short answer: 296 rows to run


ministral-8b-2512 | Short answer: 100%|███████████████████████████| 296/296 [00:18<00:00, 16.12row/s, checkpoint saved]



DONE. Workbook written to:
C:\Users\Abderrahman\Desktop\moderation llm\data eval equipe firdaws 16 juillet 2026\merged_eval_equipe_firdaws_WITH_RESPONSES.xlsx


,sheet,task,model,rows,successful,errors,empty,unparsed,accuracy
0,QCM,mcq,mistral-large-2512,589,588,0,1,0.0,0.8878
1,QCM,mcq,mistral-medium-3-5,589,588,0,1,0.0,0.8384
2,QCM,mcq,mistral-small-2603,589,588,0,1,0.0,0.8112
3,QCM,mcq,ministral-14b-2512,589,588,0,1,0.0,0.7534
4,QCM,mcq,ministral-8b-2512,589,588,0,1,0.0,0.7092
5,Binary questions,binary,mistral-large-2512,245,236,0,9,0.0,0.7288
6,Binary questions,binary,mistral-medium-3-5,245,236,0,9,0.0,0.7542
7,Binary questions,binary,mistral-small-2603,245,236,0,9,0.0,0.7119
8,Binary questions,binary,ministral-14b-2512,245,236,0,9,0.0,0.6822
9,Binary questions,binary,ministral-8b-2512,245,236,0,9,0.0,0.6737
